## Lesson Overview

**What this lesson teaches:** how to store document embeddings in a small in-memory vector index and search it for the chunks closest to a user's question.

**What's happening under the hood:**
1. Split the report into meaningful sections.
2. Embed all document chunks in one batch.
3. Store each vector beside its source text.
4. Embed a user's question.
5. Rank stored vectors by distance and return the closest chunks.

**By the end:** you will have a minimal vector store—the indexing and search layer that connects embeddings to retrieval.

# Lesson 13: Building an In-Memory Vector Index

Lesson 12 calculated similarities directly. This lesson packages that idea into a reusable `VectorIndex` that keeps vectors aligned with their original documents and can return the nearest matches.

## Where the Vector Index Fits

```text
Document → chunks → embeddings → vector index
                                      │
Question → query embedding → search ──┘
                                      ↓
                              relevant chunks
```

This example stores everything in memory. It is ideal for learning and small experiments, but its contents disappear when the kernel stops. Production systems generally use a persistent vector database.

## Setup

Install the dependencies once if needed:

```python
%pip install voyageai python-dotenv
```

Add `VOYAGE_API_KEY=...` to a `.env` file. The cell below searches both the current directory and `Claude_API_Training`. Live embedding cells make Voyage API requests and may incur a small charge.

In [1]:
import math
import os
import re
from pathlib import Path

try:
    import voyageai
except ImportError:
    voyageai = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

env_candidates = (Path('.env'), Path('Claude_API_Training/.env'))
env_path = next((path for path in env_candidates if path.exists()), None)
if env_path is not None:
    load_dotenv(env_path)

embedding_model = 'voyage-3-large'
api_key = os.getenv('VOYAGE_API_KEY')
client = voyageai.Client(api_key=api_key) if voyageai and api_key else None

if client is None:
    print('Setup incomplete: install voyageai and configure VOYAGE_API_KEY.')
else:
    print(f'Voyage client ready; model: {embedding_model}')

Voyage client ready; model: voyage-3-large


## Load and Chunk the Report

We reuse the heading-aware chunker from Lessons 11 and 12. The lookahead splits *before* each `##` heading, so the heading stays attached to its section and becomes useful retrieval context.

In [2]:
def chunk_by_section(document_text):
    return [
        section.strip()
        for section in re.split(r'(?=^##\s)', document_text, flags=re.MULTILINE)
        if section.strip()
    ]

report_candidates = (Path('report.md'), Path('Claude_API_Training/report.md'))
report_path = next((path for path in report_candidates if path.exists()), None)
if report_path is None:
    raise FileNotFoundError('Could not find report.md')

text = report_path.read_text(encoding='utf-8')
chunks = chunk_by_section(text)
print(f'Loaded {report_path} and created {len(chunks)} chunks')
print(chunks[0].splitlines()[0])

Loaded report.md and created 15 chunks
# **Annual Interdisciplinary Research Review: Cross-Domain Insights**


## Generate Embeddings

The helper accepts either one string or a list. Document chunks use `input_type='document'`; questions use `input_type='query'`. Both must use the same model so their vectors occupy the same embedding space.

In [3]:
def generate_embeddings(texts, input_type, model=embedding_model):
    if client is None:
        raise RuntimeError('Complete the Voyage setup before requesting embeddings.')
    if input_type not in {'document', 'query'}:
        raise ValueError("input_type must be 'document' or 'query'")

    is_single_text = isinstance(texts, str)
    batch = [texts] if is_single_text else list(texts)
    result = client.embed(batch, model=model, input_type=input_type)
    return result.embeddings[0] if is_single_text else result.embeddings

## The `VectorIndex`

The index stores two parallel lists: vectors and their document dictionaries. `add_vectors` validates dimensions before storing a batch. `search` computes the distance from a query vector to every stored vector; lower distance means a closer match.

Cosine distance is `1 - cosine similarity`. Euclidean distance is included to make the design extensible, although cosine is the default for this lesson.

In [4]:
class VectorIndex:
    def __init__(self, distance_metric='cosine'):
        if distance_metric not in {'cosine', 'euclidean'}:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self.vectors = []
        self.documents = []
        self.vector_dim = None
        self.distance_metric = distance_metric

    def add_vector(self, vector, document):
        if not isinstance(document, dict) or 'content' not in document:
            raise ValueError("document must be a dictionary with a 'content' key")
        if not isinstance(vector, list) or not vector:
            raise TypeError('vector must be a non-empty list')
        if not all(isinstance(value, (int, float)) for value in vector):
            raise TypeError('every vector value must be numeric')
        if self.vector_dim is None:
            self.vector_dim = len(vector)
        elif len(vector) != self.vector_dim:
            raise ValueError(f'expected {self.vector_dim} dimensions, got {len(vector)}')
        self.vectors.append(list(vector))
        self.documents.append(document)

    def add_vectors(self, vectors, documents):
        if len(vectors) != len(documents):
            raise ValueError('vectors and documents must have the same length')
        for vector, document in zip(vectors, documents):
            self.add_vector(vector, document)

    @staticmethod
    def cosine_distance(vector_a, vector_b):
        dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
        magnitude_a = math.sqrt(sum(a * a for a in vector_a))
        magnitude_b = math.sqrt(sum(b * b for b in vector_b))
        if magnitude_a == 0 or magnitude_b == 0:
            return 1.0
        similarity = dot_product / (magnitude_a * magnitude_b)
        return 1.0 - max(-1.0, min(1.0, similarity))

    @staticmethod
    def euclidean_distance(vector_a, vector_b):
        return math.sqrt(sum((a - b) ** 2 for a, b in zip(vector_a, vector_b)))

    def search(self, query_vector, k=2):
        if not self.vectors:
            return []
        if not 1 <= k <= len(self.vectors):
            raise ValueError('k must be between 1 and the number of stored vectors')
        if len(query_vector) != self.vector_dim:
            raise ValueError(f'expected {self.vector_dim} dimensions, got {len(query_vector)}')

        distance_fn = (
            self.cosine_distance
            if self.distance_metric == 'cosine'
            else self.euclidean_distance
        )
        ranked = sorted(
            [
                (distance_fn(query_vector, vector), document)
                for vector, document in zip(self.vectors, self.documents)
            ],
            key=lambda item: item[0],
        )
        return [
            {'document': document, 'distance': distance}
            for distance, document in ranked[:k]
        ]

    def __len__(self):
        return len(self.vectors)

## Embed and Index the Document Chunks

Embed the chunks as one batch, then add each vector beside a document containing its text and section number. Batching avoids one API round trip per chunk and helps reduce rate-limit pressure.

In [5]:
store = None
if client is None:
    print('Skipping live indexing. Complete the setup, then rerun this cell.')
else:
    document_embeddings = generate_embeddings(chunks, input_type='document')
    documents = [
        {'content': chunk, 'chunk_id': index}
        for index, chunk in enumerate(chunks)
    ]
    store = VectorIndex(distance_metric='cosine')
    store.add_vectors(document_embeddings, documents)
    print(f'Indexed {len(store)} chunks with {store.vector_dim} dimensions each')

Indexed 15 chunks with 1024 dimensions each


## Embed a Question and Search

A user's question is embedded as a query, then compared with every stored document vector. We request the two smallest distances. Notice that distance is a dissimilarity score: **smaller is better**.

In [6]:
question = 'What happened during the cybersecurity incident and how was it contained?'

if store is None:
    results = []
    print('No index yet. Run the live indexing cell first.')
else:
    query_embedding = generate_embeddings(question, input_type='query')
    results = store.search(query_embedding, k=2)
    for rank, result in enumerate(results, start=1):
        heading = result['document']['content'].splitlines()[0]
        print(f"{rank}. distance={result['distance']:.4f} | {heading}")

1. distance=0.4497 | ## Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011
2. distance=0.6419 | ## Section 2: Software Engineering - Project Phoenix Stability Enhancements


## Inspect the Retrieved Context

Ranking is only useful if the returned text actually contains evidence for the question. Inspect the chunks—not just their scores—before passing them to a language model.

In [7]:
if not results:
    print('No search results to inspect yet.')
else:
    for rank, result in enumerate(results, start=1):
        print(f"--- RESULT {rank} | distance={result['distance']:.4f} ---")
        print(result['document']['content'])
        print()

--- RESULT 1 | distance=0.4497 ---
## Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2023-Q4-011`. Threat intelligence indicates the activity aligns with tactics, techniques, and procedures associated with the `ShadowNet Syndicate` threat actor group. Initial access was gained via a spear-phishing email targeting personnel within the finance department, potentially seeking data relevant to Section 3 (Financial Analysis). Endpoint detection and response (EDR) systems flagged anomalous process execution (`PID: 7812`) on workstation `WS-FIN-112`. Subsequent investigation identified malware (`SHA256:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`) attempting lateral movement towards server `SRV-FIN-03`. Containment involved isolating affected systems and blocking associated command-and-control infrastructure (IP `198.51.10

## Local Sanity Checks

These tests do not call Voyage. They verify chunk structure, distance behavior, ranking order, and document-vector alignment.

In [ ]:
test_store = VectorIndex()
test_store.add_vectors(
    [[1.0, 0.0], [0.0, 1.0]],
    [{'content': 'horizontal'}, {'content': 'vertical'}],
)
test_results = test_store.search([0.9, 0.1], k=2)

assert chunks and all(chunk.strip() for chunk in chunks)
assert len(test_store) == 2
assert math.isclose(VectorIndex.cosine_distance([1, 0], [1, 0]), 0.0)
assert math.isclose(VectorIndex.cosine_distance([1, 0], [0, 1]), 1.0)
assert test_results[0]['document']['content'] == 'horizontal'
assert test_results[0]['distance'] < test_results[1]['distance']
print('All local checks passed.')

## Practice: Explore the Index

Try these one at a time and predict the outcome before running:

1. **Change the question:** search for `ERR_MEM_ALLOC_FAIL_0x8007000E`. Which section has the smallest distance?
2. **Change `k`:** return one result, then five. What does extra context add, and what irrelevant material appears?
3. **Compare metrics locally:** build a small test index with `distance_metric='euclidean'`. Does it rank your hand-written vectors the same way as cosine distance?
4. **Trigger validation:** try adding a three-dimensional vector to `test_store`. Read the error and explain why mixed dimensions cannot be compared.
5. **Add metadata:** include a source filename in each document dictionary and print it with every result.

## Summary

- A vector index keeps embeddings paired with the source documents they represent.
- Batch document embedding is more efficient than embedding one chunk at a time.
- Queries and documents use different input types but the same embedding model.
- Search ranks chunks by distance; with cosine distance, lower scores are better.
- Dimension checks prevent invalid comparisons and metadata lets results point back to their sources.
- This in-memory implementation teaches the mechanics; persistent vector databases add durability and scalable search.